Web scraping is the automated process of extracting data from websites. It's like having a robot browse the internet for you and collect specific information.

Importance of Web Scraping:
- Data Collection: It's crucial for gathering large datasets for analysis, research, and machine learning. Businesses use it for market research, competitor analysis, and lead generation.
- Monitoring: Companies can monitor product prices, news, social media trends, or competitor activities in real-time.
Content Aggregation: News aggregators, job boards, and comparison shopping sites rely on web scraping to collect and display information from various sources.
- Academic Research: Researchers use it to collect data for studies in various fields, from social sciences to economics.

Precautions and Limitations:
- Legality and Ethics: Always check a website's robots.txt file (e.g., www.example.com/robots.txt) and terms of service. Many websites prohibit scraping, and doing so without permission can lead to legal issues or your IP being blocked.
- Website Changes: Websites frequently change their structure (HTML). This can break your scraping scripts, requiring constant maintenance.
- Bot Detection: Websites use various techniques to detect and block automated scraping, such as CAPTCHAs, IP blocking, and sophisticated bot detection software. This can lead to 403 Forbidden errors, as seen in a previous cell.
- Server Load: Aggressive scraping can put a heavy load on a website's server, potentially slowing it down or even crashing it. Be considerate and implement delays between requests.
- Data Quality: Scraped data might be messy, incomplete, or contain errors, requiring significant cleaning and pre-processing.
- Dynamic Content: Many modern websites use JavaScript to load content dynamically, which can be challenging for basic scrapers. Libraries like Selenium (which we used in a previous cell) are often needed to handle such cases, as they simulate a full browser environment.
- Copyright: The data you scrape might be copyrighted. Be mindful of how you use and disseminate the information you collect.

In summary, web scraping is a powerful tool, but it must be used responsibly, ethically, and with an understanding of its technical challenges and legal implications.

Task:

We want to use some web-scraping from a website. Specifically, we want to extract PDF reports and extract their data.

We will use as example the following:

https://github.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/tree/main/input_data/certifications

Prompt:

1. Using Python extract all the PDF links from the website: https://github.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/tree/main/input_data/certifications. Create a Pandas dataframe with the following column pdf_link which for every row has each extracted link.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://github.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/tree/main/input_data/certifications"

try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    soup = BeautifulSoup(response.content, 'html.parser')

    pdf_links = []

    # Find all <a> tags
    for link in soup.find_all('a', href=True):
        href = link['href']
        # Check if the link points to a PDF file within the repository structure
        # GitHub tree view links for files often contain '/blob/' and end with the file extension.
        if '/blob/' in href and href.endswith('.pdf'):
            # Construct the raw content URL
            # Example: /fgonzaleumbc/DATA601_L09-Databases_Files_APIs/blob/main/input_data/certifications/filename.pdf
            # should become: https://raw.githubusercontent.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/main/input_data/certifications/filename.pdf
            raw_link = href.replace('/blob/', '/')
            full_pdf_url = f"https://raw.githubusercontent.com{raw_link}"
            pdf_links.append(full_pdf_url)

    # Create a Pandas DataFrame
    df = pd.DataFrame({'pdf_link': pdf_links})

    print(f"Found {len(df)} PDF links:")
    print(df.head())

except requests.exceptions.RequestException as e:
    print(f"Error fetching the URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Found 6 PDF links:
                                            pdf_link
0  https://raw.githubusercontent.com/fgonzaleumbc...
1  https://raw.githubusercontent.com/fgonzaleumbc...
2  https://raw.githubusercontent.com/fgonzaleumbc...
3  https://raw.githubusercontent.com/fgonzaleumbc...
4  https://raw.githubusercontent.com/fgonzaleumbc...


You may encounter connection errors in some cases. For whcih an alternative may be to save the file as a mhtml and then run the prompt directing to the file instead of the webpage link. We will look into this later.

You may also need to do some spot checking to refine the code. For example, there may be duplicated values and need to refine the code to remove duplicated values.

Further prompts:

1. Check if there are duplicated values.

2. Remove duplicate links from the dataframe and reset the index.


In [ ]:
print("Original DataFrame shape:", df.shape)
print("Number of duplicate links:", df.duplicated().sum())

# Remove duplicate links and reset index
df_unique = df.drop_duplicates().reset_index(drop=True)

print("DataFrame shape after removing duplicates:", df_unique.shape)
print("First 5 rows of the DataFrame with unique links:")
print(df_unique.head())

Original DataFrame shape: (6, 1)
Number of duplicate links: 3
DataFrame shape after removing duplicates: (3, 1)
First 5 rows of the DataFrame with unique links:
                                            pdf_link
0  https://raw.githubusercontent.com/fgonzaleumbc...
1  https://raw.githubusercontent.com/fgonzaleumbc...
2  https://raw.githubusercontent.com/fgonzaleumbc...


After inspecting spot checking some of the links and the dataframe is correct let's go to the next step. We want to extract the data from the PDF into a new column.

Prompt:

1. In the dataframe, create a new column called raw_text. Iterate through each row and for each link extract the pdf text and place it in the corresponding raw_text column. Be mindful and add a wait time between report download of a few seconds.

Sometimes you may encounter errors related to bot detection via Beautifoulsoup. Using other libraries like Selenium mimics how a full browser works. Sometimes the Gen AI assistant may detect this. Other times it may go into an infinite loop trying to diagnose the issue.

Be mindful that web-scraping large amounts of data may be forbidden by the robots.txt file and in some cases may be detected as a cyber security event.

In [ ]:
!pip install PyPDF2
import requests
import PyPDF2
import io
import time

def extract_text_from_pdf(pdf_url):
    try:
        # Download the PDF content
        response = requests.get(pdf_url, stream=True)
        response.raise_for_status()

        # Use BytesIO to handle the PDF content in memory
        pdf_file = io.BytesIO(response.content)

        # Create a PDF reader object
        pdf_reader = PyPDF2.PdfReader(pdf_file)

        text = []
        for page_num in range(len(pdf_reader.pages)):
            page_obj = pdf_reader.pages[page_num]
            text.append(page_obj.extract_text())
        return '\n'.join(text)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {pdf_url}: {e}")
        return None
    except PyPDF2.errors.PdfReadError as e:
        print(f"Error reading PDF {pdf_url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for {pdf_url}: {e}")
        return None

# Ensure df_unique is available from previous steps
if 'df_unique' not in locals():
    print("Error: df_unique DataFrame not found. Please run previous cells.")
else:
    # Create a new column 'raw_text' and initialize with None
    df_unique['raw_text'] = None

    # Iterate through each row to extract PDF text
    for index, row in df_unique.iterrows():
        pdf_link = row['pdf_link']
        print(f"Extracting text from: {pdf_link}")
        extracted_text = extract_text_from_pdf(pdf_link)
        df_unique.at[index, 'raw_text'] = extracted_text
        time.sleep(2) # Add a 2-second delay between requests

    print("Extraction complete. Displaying DataFrame with raw_text column:")
    print(df_unique.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.8 MB/s eta 0:00:00
Extracting text from: https://raw.githubusercontent.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/main/input_data/certifications/Google_Cloud_Learn_Certifications.pdf
Extracting text from: https://raw.githubusercontent.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/main/input_data/certifications/Microsoft_Certification-Poster_en-us.pdf
Extracting text from: https://raw.githubusercontent.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/main/input_data/certifications/nvidia-learning-training%20certifications.pdf
Extraction complete. Displaying DataFrame with raw_text column:
                                            pdf_link  \
0  https://raw.githubusercontent.com/fgonzaleumbc...   
1  https://raw.githubusercontent.com/fgonzaleumbc...   
2  https://raw.githubusercontent.com/fgonzaleumbc...   

                                            raw_text  
0                                             \n\n\n

Let's investigate why the first PDF's text extraction resulted in an empty string. We will re-run the `extract_text_from_pdf` function specifically for the first link and print the output to get a clearer picture.

In [ ]:
# Get the PDF link for the first record
first_pdf_link = df_unique['pdf_link'].iloc[0]

print(f"Re-extracting text from: {first_pdf_link}")
re_extracted_text = extract_text_from_pdf(first_pdf_link)

print("--- Full re-extracted text for the first PDF ---")
print(re_extracted_text)
print("--------------------------------------------------")

if re_extracted_text and re_extracted_text.strip():
    print("It seems some text was extracted this time.")
else:
    print("The re-extracted text is still empty or contains only whitespace.")
    print("This suggests the PDF might be image-based (scanned) or its text content is not easily parsable by PyPDF2.")
    print("If this PDF is expected to have text, an Optical Character Recognition (OCR) tool might be needed.")

Re-extracting text from: https://raw.githubusercontent.com/fgonzaleumbc/DATA601_L09-Databases_Files_APIs/main/input_data/certifications/Google_Cloud_Learn_Certifications.pdf
--- Full re-extracted text for the first PDF ---




--------------------------------------------------
The re-extracted text is still empty or contains only whitespace.
This suggests the PDF might be image-based (scanned) or its text content is not easily parsable by PyPDF2.
If this PDF is expected to have text, an Optical Character Recognition (OCR) tool might be needed.


In [ ]:
df_unique.raw_text[1]

'Role-based\nExpand your technical skill setSpecialty\nDeepen your technical skills and  \nmanage industry solutionsBusiness\nExpand your AI skills \nfor business rolesFundamentals\nMaster the basics\nDynamics 365 Business  \nCentral Functional  \nConsultant Associate\nMB-800\nDynamics 365 Business \nCentral Developer Associate \nMB-820Dynamics 365 Customer \nService Functional  \nConsultant Associate\nMB-230Azure Administrator Associate\nAZ-104Azure Security  \nEngineer Associate\nAZ-500\nAzure for SAP  \nWorkloads Specialty\nAZ-120\nAzure Virtual  \nDesktop Specialty \nAZ-140Azure Cosmos DB  \nDeveloper Specialty\nDP-420Azure Fundamentals\nAZ-900\nAzure AI Apps and Agents \nDeveloper Associate\nAI-103Azure Data Fundamentals\nDP-900\nPower Platform  \nDeveloper Associate\nPL-400Power Platform Functional\nConsultant Associate\nPL-200\nDynamics 365 Sales AI \nConsultant Associate\nAB-210Intelligent Applications \nBuilder Associate \nAB-410\nDynamics 365 Contact Center \nAI Engineer Asso